# Aula 11 - Notebook: Modelagem Topológica da Envasadora como Dígrafo Ponderado

Neste notebook implementamos a classe base `GrafoTubulacao` para representar o fluxo físico de água e copos da Máquina de Envasamento como um Grafo Dirigido e Ponderado $G=(V, E, W)$, extraindo as suas Matrizes de Adjacência para os algoritmos de roteamento.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

from typing import List, Dict, Any

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_conexao(self, origem: str, destino: str, distancia_cm: float, 
                           atuador: str, diametro_pol: float = 0.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = distancia_cm
        
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Distância (cm)": distancia_cm,
            "Atuador / Via": atuador,
            "Diâmetro (pol)": diametro_pol
        })

    def obter_graus(self) -> List[Dict[str, Any]]:
        graus = []
        for i, v in enumerate(self.vertices):
            deg_out = sum(self.adj_binaria[i])
            deg_in = sum(self.adj_binaria[r][i] for r in range(self.n))
            graus.append({"Equipamento / Nó": v, "Grau Entrada (deg-)": deg_in, "Grau Saída (deg+)": deg_out})
        return graus


In [2]:
nos_processo = [
    "TK-101_Agua", "P-101_BombaA", "P-102_BombaB", "MAN-200_Agua", 
    "DOS-201_Dosador", "BIC-202_Bico", "MAG-100_Copos", "MESA-000_Index", "EST-500_Saida"
]
rede = GrafoTubulacao(nos_processo)

# Fluxo Hidráulico (Água)
rede.adicionar_conexao("TK-101_Agua", "P-101_BombaA", 150.0, "XV-101A", 2.0)
rede.adicionar_conexao("TK-101_Agua", "P-102_BombaB", 150.0, "XV-101B", 2.0)
rede.adicionar_conexao("P-101_BombaA", "MAN-200_Agua", 80.0, "XV-200A", 2.0)
rede.adicionar_conexao("P-102_BombaB", "MAN-200_Agua", 85.0, "XV-200B", 2.0)
rede.adicionar_conexao("MAN-200_Agua", "DOS-201_Dosador", 40.0, "XV-201", 1.5)
rede.adicionar_conexao("DOS-201_Dosador", "BIC-202_Bico", 20.0, "XV-202", 1.0)
rede.adicionar_conexao("BIC-202_Bico", "MESA-000_Index", 10.0, "Gravidade", 1.0)

# Fluxo Mecânico (Copos)
rede.adicionar_conexao("MAG-100_Copos", "MESA-000_Index", 15.0, "Cilindro A", 0.0)
rede.adicionar_conexao("MESA-000_Index", "EST-500_Saida", 50.0, "Cilindro H", 0.0)

print("Tabela de Conexões da Envasadora (Água e Copos):")
print(formatar_tabela(rede.arestas_detalhes))
print("\n--- Graus Topológicos da Máquina ---")
print(formatar_tabela(rede.obter_graus()))

print("\n--- Matriz de Adjacência Ponderada (cm) ---")
print(formatar_matriz(rede.adj_pesos, rede.vertices, rede.vertices))

print("\n[OK] Matriz de Adjacência Ponderada parametrizada em memória!")
